
Here I will join all five Silver tables into a clean business-ready dataset and produce three aggregated Gold Delta tables.

Build the finalized Fact Table (claims_finalized)
Three intentional engineering details are built into this step:

**Filter is_active == True on beneficiaries before joining so SCD Type 2 history rows don't cause duplicate claim counts.

**Use F.broadcast() on the small lookup tables (providers, diagnosis_codes, procedure_codes) to avoid unnecessary shuffle joins.

**Alias overlapping columns (city, state, region exist in both beneficiaries and providers) so Spark doesn't throw ambiguous column errors.

In [0]:
from pyspark.sql import functions as F

 # 1. Read all 5 Silver Delta tables

df_claims = spark.read.format("delta").load("dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/silver/claims")
df_beneficiaries = spark.read.format("delta").load("dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/silver/beneficiary")
df_providers = spark.read.format( "delta").load("dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/silver/providers")
df_diagnosis_codes = spark.read.format("delta").load("dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/silver/diagnosis_codes")
df_procedure_codes = spark.read.format("delta").load("dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/silver/procedure_codes")

# 2. Join fact + active dimension + broadcast small lookups

claims_finalized =( 
               df_claims.alias("c")
               .join(df_beneficiaries.alias("b"), on = "beneficiary_id", how ="left")
               .join(df_providers.alias("p"), on  ="provider_id", how = "left")
               .join(F.broadcast(df_diagnosis_codes.alias("d")), on = "diagnosis_code", how = "left")
               .join(F.broadcast(df_procedure_codes.alias("pr")), on = "procedure_code", how = "left")
               .select(F.col("c.claim_id"),
        F.col("c.claim_type"),
        F.col("c.claim_status"),
        F.col("c.rejection_reason"),
        F.col("c.service_date"),
        F.date_format(F.col("c.service_date"), "yyyy-MM").alias("service_year_month"),
        F.col("c.admission_date"),
        F.col("c.discharge_date"),
        F.col("c.payment_date"),
        F.col("c.billed_amount").cast("double").alias("billed_amount"),
        F.col("c.paid_amount").cast("double").alias("paid_amount"),

        # Derived business metrics

        F.round(F.col("c.billed_amount") - F.col("c.paid_amount"), 2).alias("unpaid_amount"),
        F.datediff(F.col("c.discharge_date"), F.col("c.admission_date")).alias("length_of_stay_days"),
        F.datediff(F.col("c.payment_date"), F.col("c.service_date")).alias("payment_turnaround_days"),

        # Beneficiary details

        F.col("b.beneficiary_id"),
        F.col("b.gender"),
        F.col("b.city").alias("beneficiary_city"),
        F.col("b.state").alias("beneficiary_state"),
        F.col("b.region").alias("beneficiary_region"),

        # Provider details

        F.col("p.provider_id"),
        F.col("p.provider_name"),
        F.col("p.provider_type"),
        F.col("p.specialty"),
        F.col("p.city").alias("provider_city"),
        F.col("p.region").alias("provider_region"),

        # diagnosis codes

        F.col("d.diagnosis_code"),
        F.col("d.diagnosis_description"),
        F.col("d.diagnosis_category"),
        F.col("pr.procedure_code"),
        F.col("pr.procedure_description"),
        F.col("pr.procedure_category")
    )
)

# 3. Write finalized fact table to Gold (partitioned by claim_type)

claims_finalized.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("claim_type") \
    .save("dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/gold/claims_finalized")

print(f"claims_finalized saved! Row count: {claims_finalized.count()}")  


claims_finalized saved! Row count: 2000
